In [2]:
import mediapipe as mp
print("MediaPipe Version:", mp.__version__)
print("Does solutions exist?", hasattr(mp, 'solutions'))

MediaPipe Version: 1.2.3
Does solutions exist? False


In [ ]:
!py -3.11 -m pip uninstall mediapipe -y
!py -3.11 -m pip install mediapipe==0.10.21

Found existing installation: mediapipe 1.2.3
Uninstalling mediapipe-1.2.3:
  Successfully uninstalled mediapipe-1.2.3


In [1]:
!py -3.11 -m pip install mediapipe==0.10.21 --no-cache-dir

^C


In [4]:
!py -3.11 -m pip install mediapipe==0.10.21 -i https://pypi.tuna.tsinghua.edu.cn/simple --default-timeout=1000 --no-cache-dir

^C


In [5]:

import sys

print("Python:", sys.version)
print("Executable:", sys.executable)

packages = [
    "mediapipe",
    "cv2",
    "numpy",
    "sklearn",
    "tensorflow",
    "keras"
]

import importlib.util

for package in packages:
    print(package, ":", 
          "AVAILABLE" if importlib.util.find_spec(package)
          else "MISSING")


Python: 3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]
Executable: C:\Users\PMLS\AppData\Local\Programs\Python\Python311\python.exe
mediapipe : MISSING
cv2 : AVAILABLE
numpy : AVAILABLE
sklearn : AVAILABLE
tensorflow : AVAILABLE
keras : AVAILABLE


In [2]:
%pip install mediapipe

  Using cached mediapipe-1.2.3-py3-none-win_amd64.whl (25.7 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.3
[notice] To update, run: C:\Users\PMLS\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os

# 1. Setup MediaPipe Hands (NOT Holistic)
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.5)

def extract_landmarks(results):
    if results.multi_hand_landmarks:
        hand = results.multi_hand_landmarks[0]
        wrist_x, wrist_y, wrist_z = hand.landmark[0].x, hand.landmark[0].y, hand.landmark[0].z
        landmarks = []
        for lm in hand.landmark:
            landmarks.extend([lm.x - wrist_x, lm.y - wrist_y, lm.z - wrist_z])
        return landmarks
    return [0.0] * 63

X_data = []
y_data = []

WORDS = ["HELLO", "THANKYOU", "YES", "NO", "PLEASE"]
SEQUENCE_LENGTH = 30 

# Keep track of how many takes you have done per word
counts = {word: 0 for word in WORDS}

cap = cv2.VideoCapture(0)
print("Press 1-5 to select word | Press 's' to Record | Press 'q' to Quit")
current_word = WORDS[0]

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    frame = cv2.flip(frame, 1)
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    # Show current word and how many takes you have done
    cv2.putText(frame, f"Word: {current_word} ({counts[current_word]}/10)", (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(frame, "Press 's' to Record", (10, 60), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    cv2.imshow("Data Collection", frame)
    key = cv2.waitKey(1)

    if key == ord('1'): current_word = WORDS[0]
    if key == ord('2'): current_word = WORDS[1]
    if key == ord('3'): current_word = WORDS[2]
    if key == ord('4'): current_word = WORDS[3]
    if key == ord('5'): current_word = WORDS[4]

    if key == ord('s'):
        print(f"Recording 30 frames for {current_word}...")
        sequence = []
        for i in range(SEQUENCE_LENGTH):
            ret, frame = cap.read()
            frame = cv2.flip(frame, 1)
            res = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            sequence.append(extract_landmarks(res))
            
            cv2.putText(frame, f"Recording... {i+1}/30", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
            cv2.imshow("Data Collection", frame)
            cv2.waitKey(33) # roughly 30 FPS
        
        X_data.append(sequence)
        y_data.append(current_word)
        counts[current_word] += 1
        print(f"✅ Saved sequence {len(X_data)} for {current_word}")

    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
hands.close()

# Save the data
np.save('X_lstm_custom.npy', np.array(X_data))
np.save('y_lstm_custom.npy', np.array(y_data))
print("=====================================")
print(f"Data Collection Complete! Total sequences: {len(X_data)}")
print(f"Counts per word: {counts}")
print("Saved as X_lstm_custom.npy and y_lstm_custom.npy")